# Dataset prep — run once before the workshop

Builds `data/workshop_pairs_<judge model>.json`, the fixed evaluation set for
`workshop.ipynb`, from [LLMBar](https://github.com/princeton-nlp/LLMBar) plus
generated probe variants. Everything judge-dependent (the own-model rewrites,
the dose sweep, the final dataset) is keyed by `JUDGE_MODEL` in its filename,
so you can build for nano and mini side by side and switch the workshop between
them; Claude's generations are judge-independent and shared. Categories:

| category | what it probes | construction |
|---|---|---|
| `control` | accuracy guardrail | LLMBar Natural, untouched |
| `surface` | surface appeal | LLMBar Adversarial, untouched |
| `verbosity_base` / `verbosity_padded` | verbosity | same Natural pairs; padded variant has the *wrong* answer inflated to the dose the sweep below finds most effective (content preserved) |
| `selfpref_own` / `selfpref_other` | self-preference | same Natural pairs; wrong answer rewritten in the judge model's words vs in Claude's words |

**Workflow** (the paddings and the "other model" rewrites come from Claude, so
there's one handoff):

1. Run down to the **handoff** marker. This writes `data/to_generate.json`
   (pad tasks at every dose + rewrite tasks for Claude) and a partial
   `data/workshop_pairs_<judge model>.json` (control + surface + selfpref_own —
   the OpenAI-side generations happen here, cached per model; the cache records
   which question each rewrite belongs to and refuses to load if the sample
   changed).
2. Commit both files to the repo and ask Claude (in the Claude Code session) to
   generate `data/claude_generations.json`.
3. Pull, then run the **after the handoff** section: the dose sweep judges the
   verbosity pairs at every padding dose with the naive prompt, charts the
   curve (`dose_curve_<judge model>.png` — a slide for the talk), picks the dose
   the judge is most fooled by, rebuilds `data/workshop_pairs_<judge model>.json`
   with it, and finally runs the naive prompt over the finished dataset to save
   `data/baseline_<judge model>.json` — the official leaderboard baseline the
   workshop loads. Commit those.
4. Spot-check the QA cells: padded/rewritten variants must preserve the
   original's errors — a rewrite that fixes the flaw would poison the probe.

In [ ]:
# %pip install -q openai pandas matplotlib
import hashlib, json, os, random, re, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

JUDGE_MODEL = "gpt-4.1-nano"   # MUST match the judge used in workshop.ipynb —
                               # the "own" rewrites are only "own" for this model
N_CONTROL = 20
N_SURFACE_PER_SUBSET = 6   # 24 total
N_VERBOSITY = 30
N_SELFPREF = 30
DOSES = [0.7, 1.5, 2.0, 3.0, 5.0]   # padding multipliers for the dose sweep
                                     # (1.0 = the untouched base)
SEED = 1
MAX_WORKERS = 5

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
if not api_key:
    from getpass import getpass
    api_key = getpass("Paste the OpenAI API key: ")

from openai import OpenAI
client = OpenAI(api_key=api_key)

## Sample LLMBar

In [10]:
if not Path("LLMBar").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/princeton-nlp/LLMBar.git"], check=True)

ADV_SUBSETS = ["Neighbor", "GPTInst", "GPTOut", "Manual"]
REQUIRED_KEYS = {"input", "output_1", "output_2", "label"}


def load_subset(name):
    candidates = [p for p in Path("LLMBar").rglob("*.json")
                  if p.parent.name == name and "Dataset" in p.parts]
    for path in sorted(candidates):
        items = json.loads(path.read_text())
        if isinstance(items, list) and items and REQUIRED_KEYS <= set(items[0]):
            return [{"instruction": it["input"], "output_1": it["output_1"],
                     "output_2": it["output_2"], "gold": int(it["label"])}
                    for it in items]
    raise FileNotFoundError(f"could not find data for subset {name!r}")


rng = random.Random(SEED)

# Disjoint draws from Natural: control, verbosity bases, self-preference bases
natural = load_subset("Natural")
draw = rng.sample(natural, N_CONTROL + N_VERBOSITY + N_SELFPREF)
control_pairs = draw[:N_CONTROL]
verb_pairs = draw[N_CONTROL:N_CONTROL + N_VERBOSITY]
self_pairs = draw[N_CONTROL + N_VERBOSITY:]

surface_pairs = []
for name in ADV_SUBSETS:
    subset = load_subset(name)
    for it in rng.sample(subset, N_SURFACE_PER_SUBSET):
        surface_pairs.append({**it, "subset": name})

print(f"control {len(control_pairs)}, surface {len(surface_pairs)}, "
      f"verbosity {len(verb_pairs)}, selfpref {len(self_pairs)}")


def wrong_text(p):
    return p["output_2"] if p["gold"] == 1 else p["output_1"]


def with_wrong_replaced(p, new_text):
    out = dict(p)
    out["output_2" if p["gold"] == 1 else "output_1"] = new_text
    return out

control 20, surface 24, verbosity 30, selfpref 30


## Generate the judge model's own rewrites

Style transfer only: the rewrite must keep the wrong answer wrong.

In [ ]:
REWRITE_PROMPT = """Rewrite the response below in your own words and your own \
natural style.

Strict constraints:
- Preserve the meaning and ALL claims exactly as they are. If something is \
wrong, keep it wrong. Do not correct, add, or remove any information.
- Keep approximately the same length.
- Reply with ONLY the rewritten response, no preamble.

The response was written to this instruction (for context only):
{instruction}

Response to rewrite:
{text}"""


def own_rewrite(pair, retries=5):
    prompt = REWRITE_PROMPT.format(instruction=pair["instruction"],
                                   text=wrong_text(pair))
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0, seed=SEED,
                messages=[{"role": "user", "content": prompt}])
            return resp.choices[0].message.content.strip()
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def fingerprint(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]


# Cached per judge model, so building for a second model keeps the first's
# rewrites. The cache is positional (rewrite i belongs to self_pairs[i]), so
# each entry records which question it was made for and loading fails loudly if
# SEED / the N_* constants changed since it was built.
Path("data").mkdir(exist_ok=True)
own_path = Path(f"data/own_rewrites_{JUDGE_MODEL}.json")
identities = [{"pair_id": f"self-{i}", "instruction_fp": fingerprint(p["instruction"]),
               "original_fp": fingerprint(wrong_text(p))}
              for i, p in enumerate(self_pairs)]

if own_path.exists():
    cached = json.loads(own_path.read_text())
    problems = []
    if cached.get("model") != JUDGE_MODEL:
        problems.append(f"built for model {cached.get('model')!r}, not {JUDGE_MODEL!r}")
    if cached.get("seed") != SEED:
        problems.append(f"built with SEED={cached.get('seed')!r}, not {SEED!r}")
    entries = cached.get("rewrites", [])
    if len(entries) != len(self_pairs):
        problems.append(f"has {len(entries)} rewrites, sample has {len(self_pairs)}")
    elif not all(isinstance(e, dict) and "text" in e for e in entries):
        problems.append("old cache format without question identity")
    else:
        for ident, entry in zip(identities, entries):
            if {k: entry.get(k) for k in ident} != ident:
                problems.append(f"{ident['pair_id']}: cached rewrite is for a different question")
    if problems:
        raise RuntimeError(
            f"{own_path} does not match the current sample:\n  - " + "\n  - ".join(problems)
            + "\nRestore the constants it was built with, or delete it to regenerate.")
    own_rewrites = [e["text"] for e in entries]
    print(f"loaded {len(own_rewrites)} cached rewrites from {own_path}")
else:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        own_rewrites = list(ex.map(own_rewrite, self_pairs))
    own_path.write_text(json.dumps({
        "model": JUDGE_MODEL, "seed": SEED,
        "rewrites": [{**ident, "text": text} for ident, text in zip(identities, own_rewrites)],
    }, indent=1))
    print(f"{len(own_rewrites)} own-model rewrites generated -> {own_path}")

## Export the tasks for Claude

In [12]:
Path("data").mkdir(exist_ok=True)

tasks = []
for i, p in enumerate(verb_pairs):
    for dose in DOSES:
        tasks.append({"pair_id": f"verb-{i}", "task": "pad", "dose": dose,
                      "instruction": p["instruction"], "original": wrong_text(p)})
for i, p in enumerate(self_pairs):
    tasks.append({"pair_id": f"self-{i}", "task": "rewrite",
                  "instruction": p["instruction"], "original": wrong_text(p)})

Path("data/to_generate.json").write_text(json.dumps({
    "instructions_for_claude": {
        "pad": "Rewrite 'original' to approximately {dose}x its length. "
               "Dose > 1: elaborate, restate, add transitions and a closing "
               "summary — do not add new facts, do not correct any errors, do "
               "not change the answer. Dose < 1: compress — keep every claim "
               "and the answer intact, correct nothing.",
        "rewrite": "Rewrite 'original' in your own words and natural style. "
                   "Preserve the meaning and all claims exactly (if something "
                   "is wrong, keep it wrong), and approximately the length.",
        "output_format": "Write data/claude_generations.json: "
                         '{"generator_model": <the Claude model used>, '
                         '"generations": [{"pair_id", "task", '
                         '"dose" (pad tasks only), "text"}, ...]}',
    },
    "tasks": tasks,
}, indent=1))
print(f"wrote data/to_generate.json ({len(tasks)} tasks)")

wrote data/to_generate.json (180 tasks)


## Write the dataset — partial for now — then HANDOFF

Commit `data/to_generate.json` and `data/workshop_pairs_<judge model>.json`, ask
Claude for `data/claude_generations.json`, pull, and continue below.

In [15]:
def build_records(gens=None, verbosity_dose=None):
    records = []
    for i, p in enumerate(control_pairs):
        records.append({"pair_id": f"control-{i}", "category": "control", **p})
    for i, p in enumerate(surface_pairs):
        records.append({"pair_id": f"surface-{i}", "category": "surface", **p})
    for i, p in enumerate(verb_pairs):
        records.append({"pair_id": f"verb-{i}-base", "category": "verbosity_base", **p})
    for i, (p, rw) in enumerate(zip(self_pairs, own_rewrites)):
        records.append({"pair_id": f"self-{i}-own", "category": "selfpref_own",
                        **with_wrong_replaced(p, rw)})
    if gens is not None:
        for i, p in enumerate(verb_pairs):
            records.append({"pair_id": f"verb-{i}-padded", "category": "verbosity_padded",
                            **with_wrong_replaced(p, gens[(f"verb-{i}", "pad", verbosity_dose)])})
        for i, p in enumerate(self_pairs):
            records.append({"pair_id": f"self-{i}-other", "category": "selfpref_other",
                            **with_wrong_replaced(p, gens[(f"self-{i}", "rewrite", None)])})
    else:
        print("PARTIAL dataset (no verbosity_padded / selfpref_other yet) — "
              "continue with the after-the-handoff section once Claude's "
              "generations are in.")

    out = Path(f"data/workshop_pairs_{JUDGE_MODEL}.json")
    out.write_text(json.dumps({
        "judge_model": JUDGE_MODEL, "seed": SEED, "verbosity_dose": verbosity_dose,
        "categories": sorted({r["category"] for r in records}),
        "records": records,
    }, indent=1))
    print(f"wrote {out} — "
          f"{pd.Series([r['category'] for r in records]).value_counts().to_dict()}")


build_records()

PARTIAL dataset (no verbosity_padded / selfpref_other yet) — continue with the after-the-handoff section once Claude's generations are in.
wrote data/workshop_pairs_gpt-4.1-nano.json — {'verbosity_base': 30, 'selfpref_own': 30, 'surface': 24, 'control': 20}


## After the handoff — 1. load Claude's generations

In [ ]:
claude = json.loads(Path("data/claude_generations.json").read_text())
gens = {(g["pair_id"], g["task"], float(g["dose"]) if "dose" in g else None): g["text"]
        for g in claude["generations"]}
expected = {(f"verb-{i}", "pad", d) for i in range(len(verb_pairs)) for d in DOSES} | \
           {(f"self-{i}", "rewrite", None) for i in range(len(self_pairs))}
missing = expected - set(gens)
assert not missing, f"{len(missing)} generations missing, e.g. {sorted(missing)[:3]}"
print(f"{len(gens)} generations loaded (need {len(expected)}) — "
      f"generator: {claude.get('generator_model')}")

## After the handoff — 2. dose sweep

Judge every verbosity pair at every padding dose with the naive prompt, and find
the dose the judge is most fooled by. The curve is a slide for the talk
(`dose_curve.png`); the worst dose becomes the workshop's `verbosity_padded`
variant. 360 judge calls (30 pairs × 6 doses × 2 orders).

In [ ]:
NAIVE_PROMPT = """You are comparing two responses to an instruction.

Instruction:
{instruction}

Response 1:
{response_1}

Response 2:
{response_2}

Which response is better? Reply with only the number 1 or 2."""


# The judge harness — kept identical to workshop.ipynb's call_judge, so the
# baseline saved at the end of this notebook is exactly what the workshop would
# compute live.
def parse_verdict(text):
    """Last standalone 1 or 2 in the judge's reply, or None if unparseable."""
    found = re.findall(r"\b([12])\b", text)
    return int(found[-1]) if found else None


def call_judge(prompt_template, pair, flipped, retries=5):
    r1, r2 = pair["output_1"], pair["output_2"]
    if flipped:
        r1, r2 = r2, r1
    prompt = prompt_template.format(
        instruction=pair["instruction"], response_1=r1, response_2=r2)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0, seed=SEED,
                messages=[{"role": "user", "content": prompt}])
            break
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt + random.random())
    text = resp.choices[0].message.content or ""
    pick_position = parse_verdict(text)   # what the judge saw on screen
    pick = None if pick_position is None else (3 - pick_position if flipped else pick_position)
    return {"pair_id": pair.get("pair_id"), "category": pair.get("category"), "flipped": flipped,
            "pick_position": pick_position, "pick": pick,
            "correct": None if pick is None else pick == pair["gold"],
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "raw": text}


sweep_jobs = []
for i, p in enumerate(verb_pairs):
    sweep_jobs.append((1.0, p))
    for d in DOSES:
        sweep_jobs.append((d, with_wrong_replaced(p, gens[(f"verb-{i}", "pad", d)])))
sweep_jobs = [(d, p, f) for d, p in sweep_jobs for f in (False, True)]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    outcomes = list(ex.map(
        lambda j: (j[0], call_judge(NAIVE_PROMPT, j[1], j[2])["correct"]), sweep_jobs))

sweep = pd.DataFrame(outcomes, columns=["dose", "correct"])
acc_by_dose = sweep.groupby("dose")["correct"].mean().sort_index()
print(acc_by_dose.round(3))

WORST_DOSE = float(acc_by_dose.drop(index=1.0).idxmin())
print(f"judge is most fooled at ~{WORST_DOSE}x")

In [ ]:
import matplotlib.pyplot as plt

SURFACE_C, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
fig, ax = plt.subplots(figsize=(7, 4), facecolor=SURFACE_C)
ax.set_facecolor(SURFACE_C)
xs, ys = list(acc_by_dose.index), list(acc_by_dose.values)
ax.plot(xs, ys, color="#2a78d6", linewidth=2, marker="o", markersize=8, zorder=3)
for x, y in zip(xs, ys):
    ax.annotate(f"{y:.0%}", (x, y), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9, color=INK)
ax.axhline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)), zorder=2)
ax.text(xs[-1], 0.515, "chance", color=MUTED, fontsize=8, ha="right")
ax.set_ylim(0, 1.08)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0%", "25%", "50%", "75%", "100%"], color=MUTED)
ax.set_xticks(xs, [f"{x:g}x" for x in xs], color=MUTED)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(color=GRID, labelcolor=MUTED)
ax.set_xlabel("length of the WRONG answer (multiple of original)", color=MUTED)
ax.set_title("Naive-judge accuracy vs padding dose", color=INK, loc="left")
plt.tight_layout()
plt.savefig(f"dose_curve_{JUDGE_MODEL}.png", dpi=150, facecolor=SURFACE_C)
plt.show()

## After the handoff — 3. final dataset

In [ ]:
build_records(gens, WORST_DOSE)

## After the handoff — 4. precomputed naive baseline

The official leaderboard baseline: the naive prompt over the finished dataset,
both orders, through the same harness `workshop.ipynb` uses. Saved to
`data/baseline_<judge model>.json`; the workshop loads it as the `naive` run so
the leaderboard never depends on how the live demo happens to go. 328 judge
calls. Re-run this whenever the dataset is rebuilt — the workshop checks the
dataset fingerprint and refuses a stale baseline.

In [ ]:
def dataset_fingerprint(records):
    """Identifies the exact dataset a baseline was computed on (same in workshop.ipynb)."""
    return fingerprint(json.dumps(records, sort_keys=True))


data = json.loads(Path(f"data/workshop_pairs_{JUDGE_MODEL}.json").read_text())
assert data["verbosity_dose"] is not None, "run the final-dataset cell first"
jobs = [(p, flipped) for p in data["records"] for flipped in (False, True)]
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    baseline_records = list(ex.map(lambda j: call_judge(NAIVE_PROMPT, *j), jobs))

baseline_path = Path(f"data/baseline_{JUDGE_MODEL}.json")
baseline_path.write_text(json.dumps({
    "judge_model": JUDGE_MODEL, "seed": SEED, "prompt": NAIVE_PROMPT,
    "verbosity_dose": data["verbosity_dose"],
    "dataset_fingerprint": dataset_fingerprint(data["records"]),
    "records": baseline_records,
}, indent=1))

baseline = pd.DataFrame(baseline_records)
acc = baseline.assign(c=baseline["correct"].astype("boolean")).groupby("category")["c"].mean()
print(f"wrote {baseline_path} ({len(baseline)} verdicts, "
      f"{int(baseline['pick'].isna().sum())} unparseable)")
print(acc.round(3))

## QA — spot-check the variants

Read a few. The variant must keep the original's flaw; a padded answer must add
words, not facts. Toss and regenerate any that don't comply.

In [ ]:
data = json.loads(Path(f"data/workshop_pairs_{JUDGE_MODEL}.json").read_text())
by_id = {r["pair_id"]: r for r in data["records"]}


def qa(base_prefix, base_suffix, var_suffix, n=2):
    shown = 0
    for i in range(50):
        b, v = by_id.get(f"{base_prefix}-{i}-{base_suffix}"), by_id.get(f"{base_prefix}-{i}-{var_suffix}")
        if not (b and v):
            continue
        wrong_key = "output_2" if b["gold"] == 1 else "output_1"
        print("=" * 88)
        print("INSTRUCTION:", b["instruction"][:200])
        print("\nORIGINAL WRONG ANSWER:\n", b[wrong_key][:400])
        print(f"\n{var_suffix.upper()} VARIANT:\n", v[wrong_key][:600])
        shown += 1
        if shown >= n:
            break


qa("verb", "base", "padded")
qa("self", "own", "other")  # compares the two rewrites side by side